### Tool Binding 
Tool binding is the step where we attach the tools to llm   
so that :  
- The llm know which tools are available  
- What each of those tools do  
- What is the input schema  

In [5]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_community.tools import DuckDuckGoSearchRun

load_dotenv()

True

In [ ]:
#Define llm
llm = HuggingFaceEndpoint(
    repo_id = "deepseek-ai/DeepSeek-V4-Flash",
    task='text-generation'
)

chat_model = ChatHuggingFace(llm=llm)

In [ ]:
#Define tools
search_tool = DuckDuckGoSearchRun()

In [ ]:
#Bind tools
tooled_llm = chat_model.bind_tools([search_tool])

### Tool Calling
Tool calling is the process where LLM decides, during a conversation that it needs to call the tool - and hence it generate a structured output with:
- name of the tool
- and input arguments to call the tool with


_LLM does not actually run the tool - it only suggests which tool to run and with what arguments, further calling and getting output from tool is handled by langchain or us_

In [ ]:
tooled_llm.invoke("What is mango in 10 words?")

#Here the llm did not needed to call the tool

AIMessage(content='A sweet tropical fruit with juicy yellow flesh and a large pit.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 320, 'total_tokens': 365}, 'model_name': 'deepseek-ai/DeepSeek-V4-Flash', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f057f-7141-7b90-9bf7-9dc2034a5256-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 320, 'output_tokens': 45, 'total_tokens': 365})

In [30]:
response = tooled_llm.invoke("What are the updates on delhi pollution as on JUN 2026?")

In [31]:
print(response.content)
print(response.additional_kwargs)

#See that there is no "content" output. Tool is used in this case, now we will use this output to actually call the tool


{'tool_calls': [{'function': {'arguments': '{"query": "Delhi pollution updates June 2025 2026"}', 'name': 'duckduckgo_search', 'description': None}, 'id': 'call_00_alK31qXyXybwfa5TkRX75322', 'type': 'function'}]}


### Tool Execution
Tool execution is the step where the acutal python code (tool) is run using the input arguments that LLM suggested

There are two ways -  
1. use the arguments as input to tool OR 
2. use the whole "tool_call" as input to the tool

In [32]:
response.tool_calls[0]

{'name': 'duckduckgo_search',
 'args': {'query': 'Delhi pollution updates June 2025 2026'},
 'id': 'call_00_alK31qXyXybwfa5TkRX75322',
 'type': 'tool_call'}

In [33]:
# 1. Use arguments only
tool_result = search_tool.invoke(response.tool_calls[0]['args'])
tool_result

"3 days ago · Monthly Progress reports of June 2025 5.72 MB Download Monthly Progress reports of May 2025 8.04 MB Download Monthly Progress reports of April 2025 May 17, 2026 · Standard Operating Procedure (SOP) for Enforcement of Noise Pollution Control Measures and Implementation of Compensation Regime under the Noise Pollution (Regulation and Control) Rules, 2000 across NCT of Delhi. Apr 2, 2026 · Get latest Delhi air pollution news, daily AQI levels, smog updates, health risks, government actions, and pollution control measures in Delhi. Jun 27, 2025 · Analyse New Delhi's historical air quality trends - AQI, PM2.5, PM10, CO, SO2, NO2, O3. Understand air pollution patterns and associated health risks in the city. 1 day ago · The AQI (US) levels in New Delhi have been fluctuating throughout the last 24 hours. The levels have increased by highest 122 at 8:01 AM (Local Time), lowest 63 at 6:05 PM, Jun 24, 2026 (Local Time)."

In [34]:
# 2. Use the whole "tool_call"
tool_result = search_tool.invoke(response.tool_calls[0])
tool_result

ToolMessage(content='20 hours ago - The levels have increased by highest 122 at 8:01 AM (Local Time), lowest 71 at 12:01 AM, Jun 26, 2026 (Local Time). 5. What actions are recommended as per the current AQI (US) level in Delhi? 16 hours ago - Published : June 26, 2026 at 9:12 ... Delhi: Delhi will soon get an artificial intelligence (AI)-powered Decision Support System (DSS) to predict air pollution, identify pollution hotspots and help authorities take timely action, in collaboration ... 4 weeks ago - The average Air Quality Index (AQI) during the first five months in 2026 improved to 211. Contrast this with the figures recorded for the same period in other years: 214 (2025), 231 (2024), 213 (2023), 238 (2022), 235 (2021), 237 (2019) and ... 3 weeks ago - Skip to main content · A + A A - A A · Login · Department of Environment Government of NCT of Delhi · Air Pollution Mitigation Plan 2025 · Home/ Environment/ Air Pollution Mitigation Plan 2025 · Date: 16-06-2025 | 2090.93 KB · Downlo

I can see DuckDuckGo is not so good for web search hmmmmmmmmmm Anyways.

The output is a `"ToolMessage"`, it is similar to "ai message", "human message", "system message" in prompting.  
We can give this output to LLM again as "ToolMessage" - a message from execution of a tool

So the final workflow is like:

___Tool Creation -> Tool Binding -> Tool Calling -> Tool Execution___

### InjectedToolArg